4.1 An urban mobility company operates a fleet of 7,500 shared electric scooters across a large U.S. city. Each scooter’s battery and motor assembly deteriorates with use and must eventually be replaced. On average, a scooter operates for about eight months before failure, which can be modeled as an exponential lifetime distribution. The company faces two maintenance options: replace components individually when failures occur, or schedule group replacement rounds at regular time intervals. Individual replacements cost $310 per scooter, including parts, labor, and downtime, while group replacements cost 192 per scooter but require a coordinated maintenance event across the fleet. 

The operations team wants to determine the optimal interval between group replacements that minimizes the long-run maintenance cost. Assume that failures can occur at any time between group rounds and that scooters that fail early are replaced individually and then included again in the next scheduled round. Using this information, simulate the average monthly cost per scooter under both the individual and group policies, and identify which policy is optimal. 



In [ ]:
# Import Necessary Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Problem imputs
fleet = 7500
mean_life_months = 8
lam = 1 / mean_life_months          # exponential failure rate per month
individual_cost = 310
group_cost = 192
rng = np.random.default_rng(42)

In [ ]:
# Simulate individual-only policy
def simulate_individual_policy(months=240, reps=10000):
    # Over a time horizon, failures for the whole fleet follow a Poisson process
    total_failures = rng.poisson(fleet * lam * months, size=reps)
    total_cost = individual_cost * total_failures
    monthly_cost_per_scooter = total_cost / (fleet * months)
    return monthly_cost_per_scooter.mean(), monthly_cost_per_scooter.std()

In [ ]:
# Simulate group replacement policy
def simulate_group_policy(T, total_months=240, reps=10000):
    """
    T = group replacement interval in months
    Any failures before the scheduled round are replaced individually.
    At each scheduled round, all scooters are replaced as a group.
    """
    cycles = total_months // T
    if cycles == 0:
        raise ValueError("T is too large for the simulation horizon.")

    # In each cycle of length T, failures across the fleet are Poisson(fleet * lam * T)
    failures_per_cycle = rng.poisson(fleet * lam * T, size=(reps, cycles))
    total_failures = failures_per_cycle.sum(axis=1)

    total_cost = (
        individual_cost * total_failures
        + group_cost * fleet * cycles
    )

    monthly_cost_per_scooter = total_cost / (fleet * cycles * T)
    return monthly_cost_per_scooter.mean(), monthly_cost_per_scooter.std()

In [ ]:
# Run simulation
ind_mean, ind_sd = simulate_individual_policy()

results = []
for T in range(1, 25):  # test group intervals from 1 to 24 months
    sim_mean, sim_sd = simulate_group_policy(T)
    
    # theoretical long-run average monthly cost per scooter
    theoretical = group_cost / T + individual_cost * lam
    
    results.append({
        "Group interval T (months)": T,
        "Simulated monthly cost / scooter": sim_mean,
        "Simulation SD": sim_sd,
        "Theoretical monthly cost / scooter": theoretical
    })

df = pd.DataFrame(results)

print("Individual-only policy")
print(f"Simulated monthly cost per scooter = ${ind_mean:.4f}")
print()

print("Group replacement policies")
print(df.to_string(index=False))

best_group_row = df.loc[df["Simulated monthly cost / scooter"].idxmin()]
print("\nBest group policy tested:")
print(best_group_row)

print("\nOverall conclusion:")
if ind_mean < best_group_row["Simulated monthly cost / scooter"]:
    print("Individual replacement is optimal.")
else:
    print("A group replacement policy is optimal.")

## Replacing scooters only when they fail is the most cost-efficient strategy.

4.2 Building up on the early retirement analysis, compute:  

* The value of the portfolio at the time that financial independence is achieved 
* The value of the portfolio by the end of the simulation if financial independence is not achieved 
* The value of the portfolio if you decided not to retire early 

In [ ]:
# Import Necessary Libraries
import numpy as np
import pandas as pd
import scipy.stats as sp
import matplotlib.pyplot as plt

In [ ]:
# From Lecture

def summarystats(Outp):
    '''Generic code to compute descriptive statistics for each column in a data frame Outp'''
    Mean = Outp.mean()
    StDev = Outp.std(ddof=1)
    Percent05 = Outp.quantile(0.05)
    Percent95 = Outp.quantile(0.95)
    StdErr = Outp.sem()
    Min = Outp.min()
    Max = Outp.max()

    Prob = 0.95
    Tails = (1 - Prob)/2
    z = sp.norm.ppf(1-Tails)
    CI_low = Mean - z * StdErr
    CI_high = Mean + z * StdErr

    Sample_stats = pd.concat(
        [Mean, StDev, Percent05, Percent95, StdErr, CI_low, CI_high, Min, Max],
        axis=1
    )
    Sample_stats.columns = [
        'Sample Mean', 'Sample St Dev', 'Percentile 5', 'Percentile 95',
        'Std Error', 'CI low', 'CI high', 'Min', 'Max'
    ]
    return Sample_stats.round(3)


def descriptivehistogram(Outp):
    '''Plot a histogram for each column in the output dataframe'''
    for column in list(Outp.columns):
        plt.clf()
        plt.hist(Outp[column].dropna(), bins='auto', edgecolor='black', density=True)
        plt.xlabel(column)
        plt.ylabel('Frequency')
        plt.title(f'Histogram of {column}')
        plt.show()

In [ ]:
def simulationfunction_extended(controls, inputs):
    '''
    Extended version of the early retirement simulation.

    Additional outputs:
    1) Portfolio value at the time FI is achieved
    2) Portfolio value at the end of the simulation if FI is NOT achieved
    3) Portfolio value at the end of the simulation if the investor NEVER retires early
    '''
    # A. Standardized controls for the simulation
    seed = controls[0]
    n = controls[1]

    # B. Model-dependent inputs
    Y = inputs[0]          # annual contribution at the beginning of each year
    damp = inputs[1]       # damping factor
    x_input = inputs[2]    # list of allocation vectors
    hist_df = inputs[3]    # historical returns dataframe
    E = inputs[4]          # annual expenses
    m = inputs[5]          # expense multiplier
    max_years = inputs[6]  # maximum years in the simulation

    n_portfolios = len(x_input)

    # C. Use a seed or not
    if seed:
        np.random.seed(100)

    # D. Prepare historical-sampling probabilities
    hist_len = len(hist_df)
    prob = damp ** np.arange(hist_len)
    prob = prob / prob.sum()
    prob = prob[::-1]  # highest probability on most recent years

    cols = hist_df.columns.tolist()
    asset_cols = cols[1:-1]   # assumes: first col = year, last col = inflation
    inflation_col = cols[-1]

    target_pv = m * E

    # Store outputs
    years_to_FI = np.full((n, n_portfolios), max_years, dtype=int)
    A_FI = np.zeros((n, n_portfolios), dtype=int)

    # New requested outputs
    V_at_FI = np.full((n, n_portfolios), np.nan)              # nominal portfolio value when FI is first reached
    V_end_if_no_FI = np.full((n, n_portfolios), np.nan)       # nominal ending value only for trials with no FI
    V_if_no_early_retire = np.zeros((n, n_portfolios))        # nominal ending value after max_years regardless of FI

    # Optional: real/PV versions in case you want them too
    PV_at_FI = np.full((n, n_portfolios), np.nan)
    PV_end_if_no_FI = np.full((n, n_portfolios), np.nan)
    PV_if_no_early_retire = np.zeros((n, n_portfolios))

    for sim in range(n):
        # Case 1: retire as soon as FI is achieved
        V_retire = np.zeros(n_portfolios, dtype=float)
        A = np.zeros(n_portfolios, dtype=int)
        D_retire = 1.0

        # Case 2: never retire early; keep investing through full horizon
        V_continue = np.zeros(n_portfolios, dtype=float)
        D_continue = 1.0

        for t in range(1, max_years + 1):
            # draw one historical year for this simulated year
            idx = np.random.choice(range(hist_len), p=prob)
            row = hist_df.iloc[idx]

            asset_returns = row[asset_cols].values.astype(float)[:4]
            i = float(row[inflation_col])

            denom = 1.0 + i
            if denom == 0:
                denom = 1e-12

            # update discount factors
            D_retire = D_retire / denom
            D_continue = D_continue / denom

            for p_idx, alloc in enumerate(x_input):
                P = 1.0 + np.dot(asset_returns, alloc)

                # Path 1: stop contributing after FI
                if A[p_idx] == 0:
                    V_retire[p_idx] = (V_retire[p_idx] + Y) * P
                    Zt = V_retire[p_idx] * D_retire

                    if Zt >= target_pv:
                        A[p_idx] = 1
                        A_FI[sim, p_idx] = 1
                        years_to_FI[sim, p_idx] = t
                        V_at_FI[sim, p_idx] = V_retire[p_idx]
                        PV_at_FI[sim, p_idx] = Zt

                # Path 2: never retire early, always contribute to max_years
                V_continue[p_idx] = (V_continue[p_idx] + Y) * P

        # After full horizon, fill ending values
        for p_idx in range(n_portfolios):
            V_if_no_early_retire[sim, p_idx] = V_continue[p_idx]
            PV_if_no_early_retire[sim, p_idx] = V_continue[p_idx] * D_continue

            if A[p_idx] == 0:
                V_end_if_no_FI[sim, p_idx] = V_retire[p_idx]
                PV_end_if_no_FI[sim, p_idx] = V_retire[p_idx] * D_retire

    # Convert to dataframes with readable labels
    portfolio_labels = [f'Portfolio_{j+1}' for j in range(n_portfolios)]

    years_to_FI = pd.DataFrame(years_to_FI, columns=portfolio_labels)
    A_FI = pd.DataFrame(A_FI, columns=portfolio_labels)
    V_at_FI = pd.DataFrame(V_at_FI, columns=portfolio_labels)
    V_end_if_no_FI = pd.DataFrame(V_end_if_no_FI, columns=portfolio_labels)
    V_if_no_early_retire = pd.DataFrame(V_if_no_early_retire, columns=portfolio_labels)

    PV_at_FI = pd.DataFrame(PV_at_FI, columns=portfolio_labels)
    PV_end_if_no_FI = pd.DataFrame(PV_end_if_no_FI, columns=portfolio_labels)
    PV_if_no_early_retire = pd.DataFrame(PV_if_no_early_retire, columns=portfolio_labels)

    return (
        years_to_FI,
        A_FI,
        V_at_FI,
        V_end_if_no_FI,
        V_if_no_early_retire,
        PV_at_FI,
        PV_end_if_no_FI,
        PV_if_no_early_retire
    )

In [ ]:
# Run with existing imputs

# Simulation control parameters
controls = [True, 1000]

# Load historical asset returns & inflation into a data frame
file_name = r'data\'
Asset_df = pd.read_excel(file_name)

# Model inputs
inputs = [
    40000,                                  # Y
    0.97,                                   # damp
    [[0.1,0.1,0.7,0.1],
     [0.2,0.2,0.4,0.2],
     [0.3,0.3,0.1,0.3]],                    # x
    Asset_df,
    45000,                                  # E
    25,                                     # m
    45                                      # max_years
]

(
    years_to_FI,
    A_FI,
    V_at_FI,
    V_end_if_no_FI,
    V_if_no_early_retire,
    PV_at_FI,
    PV_end_if_no_FI,
    PV_if_no_early_retire
) = simulationfunction_extended(controls, inputs)

print("Years to Financial Independence")
display(summarystats(years_to_FI))

print("Indicator for Achieving Financial Independence")
display(summarystats(A_FI))

print("Portfolio Value at the Time FI is Achieved")
display(summarystats(V_at_FI))

print("Portfolio Value by the End of the Simulation if FI is NOT Achieved")
display(summarystats(V_end_if_no_FI))

print("Portfolio Value if You Decide NOT to Retire Early")
display(summarystats(V_if_no_early_retire))

More aggressive portfolios achieve financial independence faster and produce significantly higher long-term wealth, while conservative portfolios delay retirement and carry a higher risk of not reaching financial independence despite larger balances at the time it is achieved.